## Create Table to summarize results

This notebook creates a table with the normalized NLLHs for different models. It expects to find pickled predictions for 10 folds with the following filenames:

 * ```<scenario_name>.floc.<dist_name>.<fold>.50.<model>.pkl```

`eval_model.py` automatically creates such files. 

**NOTE:** The first cell contains some variables that need to be adjusted. Evaluating all scenarios might take a while.

In [ ]:
import os
import pickle
import sys

import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np
import scipy.stats as scst
from sklearn.model_selection import KFold
from sklearn.metrics import mean_squared_error
import tabulate

#% matplotlib inline

sys.path.append("../")
from helper_DistNet import load_data_continuous, preprocess

mean_exp = []  #["rf", "nn"]

# 1) Select the distribution you want to fit 
# floc_exp = ["lognormal_distfit", "lognormal_distfit.rf2", "lognormal_distfit.rf", "lognormal_nn"]
# distribution_hndl = scst.distributions.lognorm
# floc_exp = ["expon_distfit", "expon_distfit.rf2", "expon_distfit.rf", "expon_nn"]
# distribution_hndl = scst.distributions.expon
floc_exp = ["invgauss_distfit", "invgauss_distfit.rf2", "invgauss_distfit.rf", "invgauss_nn"]
distribution_hndl = scst.distributions.invgauss
# floc_exp = ["weibull_distfit", "weibull_distfit.rf2", "weibull_distfit.rf", "weibull_nn"] #, "weibull_xgb"]
# distribution_hndl = scst.distributions.weibull_min

# 2) Change the following to existing paths
print(os.getcwd())
output_dir = "../scripts_DistNet/TEST_50/"
save_dir = "./results_eval"
os.makedirs(save_dir, exist_ok=True)

# 3) Select whether to compute results for all scenarios or just a single scenario
SCENARIOS = ["DE", "DiagonalCMA", "modcma", "modde", "NaiveIsoEMNA", "NGOpt14", "NGOpt38", "OnePlusOne", "PSO", "RandomSearch", "RCobyla"]
DIMS =  [5, 20]
BUDGETS = [100, 250, 500, 1000] # , 2500, 5000, 10000]
RUNS = 50
SEED = 50
LOG_TARGET = False

load = False

In [ ]:
res_dict = dict()
for fl in os.listdir(output_dir):
    if not f".{SEED}." in fl:
        # Select only one seed
        continue
    print(fl, output_dir)

    fl_path = os.path.join(output_dir, fl)
    
    if os.path.isdir(fl_path):
        continue
    
    if not "pkl" in fl_path:
        continue
    
    if "txt" in fl_path or "png" in fl_path:
        continue

    if not os.path.isfile(fl_path):
        fl_path = os.path.join(output_dir)

    with open(fl_path, "rb") as fh:
        pkl = pickle.load(fh)
        train_pred = np.array(pkl[0])
        val_pred = np.array(pkl[1])
        add_info = pkl[2]
        task = add_info["task"]
        model = add_info["model"]
        scenario = add_info["scenario"]
        dim = add_info["dim"]
        budget = add_info["budget"]
        fold = add_info["fold"]

        if "nn" in add_info["model"] and add_info["task"] == "floc":
            print(f"{scenario}_{dim}D_{budget}, {fold}, {add_info['loaded']}")
    
    SC = f"{scenario}_{dim}D_{budget}"
    if SC not in res_dict:
        res_dict[SC] = dict()

    if model not in floc_exp and model not in mean_exp:
        continue
    
    if task not in res_dict[SC]:
        res_dict[SC][task] = dict()
        
    if model not in res_dict[SC][task]:
        res_dict[SC][task][model] = dict()

    res_dict[SC][task][model][fold] = (train_pred, val_pred)


if len(SCENARIOS) == 0 or len(DIMS) == 0 or len(BUDGETS) == 0:
    SCENARIOS = [key.split("_")[0] for key in res_dict.keys()]
    DIMS = list(set([int(key.split("_")[1][:-1]) for key in res_dict.keys()]))
    BUDGETS = list(set([int(key.split("_")[2]) for key in res_dict.keys()]))

In [ ]:
for scen in sorted(SCENARIOS):
    for dim in sorted(DIMS):
        for budget in sorted(BUDGETS):
            SC = f"{scen}_{dim}D_{budget}"
            print("### %s" % SC)
            try:
                for model in sorted(res_dict[SC]["mean"]):
                    print("mean %20s" % model, len(res_dict[SC]["mean"][model]))
            except:
                pass
            for model in sorted(res_dict[SC]["floc"]):
                print("floc %20s" % model, len(res_dict[SC]["floc"][model]))

In [ ]:
nllh_dict = dict()
mean_dict = dict()
var_dict = dict()
if load:
    with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "rb") as fh:
        nllh_dict = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "rb") as fh:
        mean_dict = pickle.load(fh)
        
    with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "rb") as fh:
        var_dict = pickle.load(fh)
    print("LOADED")

else:
    for algo in sorted(SCENARIOS):
        for dim in sorted(DIMS):
            for budget in sorted(BUDGETS):
                scen = f"{algo}_{dim}D_{budget}"
                nllh_dict[scen] = dict()
                mean_dict[scen] = {"true": {"train": [], "val": []}, "floc": dict(), "mean": dict()}
                var_dict[scen] = {"true": {"train": [], "val": []}, "floc": dict(), "mean": dict()}

                print("##### %s #####" % scen)
                # 4) Load data and compute NLLH

                target_precs, features, _ = load_data_continuous.get_data(
                    algo=algo, 
                    budget=budget, 
                    dim=dim, 
                    runs=RUNS, 
                    agregate_runs=False, 
                    log_target=LOG_TARGET)

                # We do not need any features
                del features

                # Get CV splits
                idx = list(range(target_precs.shape[0]))
                kf = KFold(n_splits=10, shuffle=True, random_state=0)
                fold = -1
                for train, valid in kf.split(idx):
                    fold += 1
                    #print("## Split %d" % ctr)
                    
                    y_tra_run = target_precs[train]
                    y_val_run = target_precs[valid]
                    
                    for i in range(y_tra_run.shape[0]):
                        y_max_ = np.max(y_tra_run[i, :])
                        y_min_ = 0

                        y_tra_run[i, :] = (y_tra_run[i, :] - y_min_) / y_max_
            
                    for i in range(y_val_run.shape[0]):
                        y_max_ = np.max(y_val_run[i, :])
                        y_min_ = 0
                        
                        y_val_run[i, :] = (y_val_run[i, :] - y_min_) / y_max_

                    task = "floc"
                    for model in floc_exp:
                        if model not in res_dict[scen][task]:
                            continue

                        if model not in nllh_dict[scen]:
                            nllh_dict[scen][model] = np.zeros([10, 2]) * np.nan
                            mean_dict[scen][task][model] = {"train": list(), "val": list()}
                            var_dict[scen][task][model] = {"train": list(), "val": list()}

                        if fold not in res_dict[scen][task][model]:
                            print("FAILED, %s, %s, %s, fold %d" % (scen, task, model, fold))
                            continue

                        for observations, data, idx in ([y_tra_run, "train", 0], [y_val_run, "val", 1]):
                            # print(res_dict[scen][task][model][fold][idx])
                            assert observations.shape[0] == res_dict[scen][task][model][fold][idx].shape[0]
                            nllh = list()
                            mean = list()
                            var = list()
                            for obs, p in zip(observations, res_dict[scen][task][model][fold][idx]):
                                if distribution_hndl.name == "expon":
                                    if type(p) == np.float64: 
                                        p = [p, ]
                                    if len(p) == 1: p = [0, p[0]]
                                    assert p[0] == 0
                                else:
                                    if len(p) == 2: p = [p[0], 0, p[1]]
                                    assert p[1] == 0
                                nllh_per_inst = distribution_hndl.logpdf(obs, *p[:-2], loc=p[-2], scale=p[-1]) + np.log(max(obs))              
                                nllh_per_inst = np.mean(-nllh_per_inst)
                                nllh.append(nllh_per_inst)
                                mean.append(distribution_hndl.mean(*p[:-2], loc=p[-2], scale=p[-1]))
                                var.append(distribution_hndl.var(*p[:-2], loc=p[-2], scale=p[-1]))
                            nllh_dict[scen][model][fold, idx] = np.mean(nllh)
                            mean_dict[scen][task][model][data].append(mean)
                            var_dict[scen][task][model][data].append(var)

    with open(save_dir + "/" + distribution_hndl.name + ".nllh.pkl", "wb") as fh:
        pickle.dump(nllh_dict, fh)

    with open(save_dir + "/" + distribution_hndl.name + ".mean.pkl", "wb") as fh:
        pickle.dump(mean_dict, fh)

    with open(save_dir + "/" + distribution_hndl.name + ".var.pkl", "wb") as fh:
        pickle.dump(var_dict, fh)
    print("dumped to " + save_dir)

In [ ]:
# Generate NLLH table

# nan means there are some folds missing
# inf means there was no results file found
tab_data = list()
header = ["sc"] + floc_exp
CAP_NUM = 1e20
for scen in sorted(nllh_dict.keys(), key=lambda x: int(x.split("_")[2])):
    dummy_array = np.ones([2, 2]) * np.inf
    tab_data.append([scen] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) 
                              if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
                              else np.inf
                              for m in header[1:]])
    tab_data.append(["", ] + [np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) 
                              if np.mean(nllh_dict[scen].get(m, dummy_array)[:, 1]) < CAP_NUM and np.mean(nllh_dict[scen].get(m, dummy_array)[:, 0]) != np.nan
                              else np.inf
                              for m in header[1:]])

a = tabulate.tabulate(tab_data, headers=header, floatfmt="5.3f")
print(a)

with open("%s/nllh_%s.txt" % (save_dir, distribution_hndl.name), "w") as fh:
    fh.write(a)